# Legal Contract Analyzer — Llama 3.1 8B via Groq (No Fine-Tuning)

This notebook analyzes legal contracts using RAG + Llama 3.1 8B served for free via Groq.

**Setup (one time):**
1. Go to https://console.groq.com and create a free account
2. Get your API key from the dashboard
3. Paste it in the cell below

**Pipeline:**
```
PDF → Text extraction → Chunking → Embedding (BGE) → FAISS index
  → Retrieve top-k chunks → Llama 3.1 8B on Groq → Answer
```


In [ ]:
# Install dependencies
!pip -q install pypdf faiss-cpu sentence-transformers groq

In [ ]:
import os

# --- Paste your Groq API key here ---
# Get one free at https://console.groq.com
os.environ['GROQ_API_KEY'] = 'YOUR_GROQ_API_KEY_HERE'

GROQ_MODEL = 'llama-3.1-8b-instant'  # free on Groq
# Other good free options on Groq:
# 'llama-3.3-70b-versatile'  -- much stronger, higher quality answers
# 'llama3-8b-8192'           -- older but stable


In [ ]:
# Upload your PDF contract
from google.colab import files
uploaded = files.upload()
pdf_path = next(iter(uploaded))
print(f"Loaded: {pdf_path}")

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() or ""

print(f"Extracted {len(full_text):,} characters across {len(reader.pages)} pages")

In [ ]:
def chunk_text(text, size=1500, overlap=200):
    """Split text into overlapping chunks."""
    text = " ".join(text.split())  # normalise whitespace
    chunks, i = [], 0
    while i < len(text):
        chunks.append(text[i:i + size])
        i += size - overlap
    return chunks

chunks = chunk_text(full_text)
print(f"Created {len(chunks)} chunks")

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("Loading embedding model (BGE base)...")
embedder = SentenceTransformer('BAAI/bge-base-en-v1.5')

print("Embedding chunks...")
embeddings = embedder.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True
).astype('float32')

# Build FAISS index
index = faiss.IndexFlatIP(embeddings.shape[1])  # inner product = cosine for normalised vectors
index.add(embeddings)
print(f"Index built with {index.ntotal} vectors")

def retrieve(query, k=5):
    """Return top-k most relevant chunks for a query."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q_emb, k)
    return [(chunks[i], float(scores[0][rank])) for rank, i in enumerate(indices[0])]

In [ ]:
from groq import Groq

groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])

SYSTEM_PROMPT = """You are a legal contract analyst. Your job is to answer questions about contracts accurately.

Rules:
- Answer ONLY from the provided context. Do not use outside knowledge.
- Always cite the specific clause, section, or party name your answer is based on.
- If the answer cannot be found in the context, say: "This information is not present in the provided contract sections."
- Be precise. Legal language matters — do not paraphrase obligations as permissions or vice versa.
- If a question asks about obligations, distinguish between 'shall' (mandatory) and 'may' (optional)."""

def ask(question, k=5, verbose=False):
    """
    Ask a question about the uploaded contract.
    
    Args:
        question: Your question about the contract
        k: Number of chunks to retrieve (increase for complex questions)
        verbose: If True, also print the retrieved context
    """
    # Retrieve relevant chunks
    results = retrieve(question, k=k)
    context = "\n\n---\n\n".join([chunk for chunk, score in results])

    if verbose:
        print("=== Retrieved context ===")
        for i, (chunk, score) in enumerate(results):
            print(f"[Chunk {i+1}, relevance={score:.3f}]\n{chunk[:300]}...\n")
        print("=" * 40)

    # Call Llama 3.1 8B on Groq
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Question: {question}\n\nContract sections:\n{context}"
            }
        ],
        temperature=0.0,   # deterministic — important for legal Q&A
        max_tokens=512,
    )

    return response.choices[0].message.content.strip()


print("Ready. Use ask('your question') to analyze the contract.")

In [ ]:
# --- Example questions ---
# Uncomment the ones you want to run, or write your own.

print(ask("Who are the parties to this contract?"))

In [ ]:
print(ask("What are the termination rights and notice periods?"))

In [ ]:
print(ask("What are the payment obligations, amounts, and due dates?"))

In [ ]:
print(ask("What confidentiality obligations exist, and how long do they last?"))

In [ ]:
print(ask("Are there any indemnification or liability clauses? What are the limits?"))

In [ ]:
# --- Structured summary of the whole contract ---

SUMMARY_QUESTIONS = [
    "Who are the parties to this contract?",
    "What is the purpose and subject matter of the contract?",
    "What is the contract duration and any renewal terms?",
    "What are the payment obligations, amounts, and due dates?",
    "What are the termination rights and required notice periods?",
    "What confidentiality obligations exist?",
    "What are the governing law and dispute resolution provisions?",
    "Are there any liability caps or indemnification clauses?",
]

print("=" * 60)
print("CONTRACT SUMMARY")
print("=" * 60)
for question in SUMMARY_QUESTIONS:
    print(f"\n{question}")
    print("-" * len(question))
    print(ask(question))
    print()

In [ ]:
# --- Interactive Q&A loop ---
# Run this cell to ask questions interactively.

print("Type your question and press Enter. Type 'quit' to stop.\n")
while True:
    q = input("Your question: ").strip()
    if q.lower() in ('quit', 'exit', 'q', ''):
        break
    print("\nAnswer:")
    print(ask(q))
    print()